In [20]:
import fastf1 as ff
import numpy as np

# Stocker le cache
ff.Cache.enable_cache("donneescachef1")

# Charger la course spécifique (Bahrein, 2024) + données pilote
session = ff.get_session(2024, "Bahrein", "R")  
session.load()
laps_ver = session.laps.pick_driver("VER")
clean_laps = laps_ver.pick_quicklaps().pick_track_status("1") # On garde uniquement les tours sous drapeau vert 
chronos_secondes = clean_laps["LapTime"].dt.total_seconds().dropna() # Convertir en secondes pour une lecture + simple

# Calcul des paramètres de la loi Normale
mu_pilote = np.mean(chronos_secondes) # Moyenne des temps du pilote
sigma_pilote = np.std(chronos_secondes) # L'ecart type

print(f"Pilote : Max VERSTAPPEN (Bahrein, 2024)")
print(f"Nb de tours clean : {len(chronos_secondes)}")
print(f"Temps de base mu : {mu_pilote:.4f} sec")
print(f"Régularité sigma : {sigma_pilote:.4f} sec")


events      WARNING 	Correcting user input 'Bahrein' to 'Bahrain Grand Prix'
core           INFO 	Loading data for Bahrain Grand Prix - Race [v3.8.3]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['1', '11', '55', '16', '63', '4', '44', '81', '14', '18', '24', '20', '3', '22', '23', '27', '31', '1

Pilote : Max VERSTAPPEN (Bahrein, 2024)
Nb de tours clean : 50
Temps de base mu : 95.5997 sec
Régularité sigma : 1.0404 sec


/Users/maximilien/PERSO 🙈/PROJET-QUANT-FINANCE/PROJET-GITHUB/.venv/lib/python3.12/site-packages/fastf1/core.py:3175: FutureWarning: pick_driver is deprecated and will be removed in a future release. Use pick_drivers instead.
  warnings.warn(("pick_driver is deprecated and will be removed"


In [21]:
import pandas as pd 

# Préparation des variables t_base, k_fuel(N-L) et a(L) pour la régression

df_reg = pd.DataFrame() # On crée un tableau vide pour la régression

df_reg["LapTime_Y"] = clean_laps["LapTime"].dt.total_seconds() # Variable pour le chronos en secondes

df_reg["L"] = clean_laps["LapNumber"] # Variable de repère temporel L, le numéro de tours

N = 57 # Distance total de Bahrein (N) = 57 tours
df_reg["Fuel_Remaining_X1"] = N - df_reg['L'] # Variable pour le carburant restant (en tours)

df_reg["Tyre_Age_X2"] = clean_laps["TyreLife"] # Variable pour l'âge du pneu, récup dans fastf1 (en tours)

df_reg = df_reg.dropna() # removing missing value 

print(df_reg.head())

   LapTime_Y    L  Fuel_Remaining_X1  Tyre_Age_X2
1     96.296  2.0               55.0          5.0
2     96.753  3.0               54.0          6.0
3     96.647  4.0               53.0          7.0
4     97.173  5.0               52.0          8.0
5     97.092  6.0               51.0          9.0


In [22]:
import statsmodels.api as sm 

# On sépare les variables (X) explicatives de la cible (Y)
X = df_reg[["Fuel_Remaining_X1", "Tyre_Age_X2"]]
Y = df_reg["LapTime_Y"]

X = sm.add_constant(X) # On ajoute notre constante, t_base

# Création et résolution du modèle OLS
modele = sm.OLS(Y, X)
resultats = modele.fit()

print(resultats.summary()) # Affichage du rapport statistique complet 

# Exract propre pour le projet 
t_base = resultats.params["const"]
k_fuel = resultats.params["Fuel_Remaining_X1"]
rho = resultats.params["Tyre_Age_X2"]


print(f"t_base (Rythme absolu sans contrainte) : {t_base:.4f}s")
print(f"k_fuel (gain lié à l'essence/tour) : {k_fuel:.4f}s")
print(f"rho (Perte lié à la gomme par tour) : {rho:.4f}s")

                            OLS Regression Results                            
Dep. Variable:              LapTime_Y   R-squared:                       0.778
Model:                            OLS   Adj. R-squared:                  0.768
Method:                 Least Squares   F-statistic:                     82.25
Date:                Fri, 24 Jul 2026   Prob (F-statistic):           4.46e-16
Time:                        10:06:12   Log-Likelihood:                -35.327
No. Observations:                  50   AIC:                             76.65
Df Residuals:                      47   BIC:                             82.39
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                        coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------
const                93.0244      0.23

In [27]:
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt

# Simulation Mont Carlo 

# Paramètres de base 
N_tours = 57
temps_stands = 24.0 # temps perdu pour un arret au stand 

# Paramètres de la regression
t_base = 93.0244
k_fuel = 0.0554
rho = 0.1000

sigma_pilote = 1.04 # l'écart type 

# Fonction de la course 
def simuler_course(tour_arret): # On veut simuler une course avec un arret au tour "tour_arret" et calculer le temps (on pourra par la suite ajouter des arrets au stand)
    temps_total_course = 0.0
    for L in range(1, N_tours + 1): # Calculer l'age du pneu comme une fonction a sauts
        if L <= tour_arret:
            age_pneu = L
        else:
            age_pneu = L - tour_arret
        chrono_theorique = t_base + (k_fuel * (N_tours - L)) + (rho * age_pneu) # l'equation de base sans le facteur humain 
        erreur_pilote = np.random.normal(0, sigma_pilote) # Calcul de l'erreur humaine a l'aide de la Loi Normale
        chrono_reel = chrono_theorique + erreur_pilote # Le chrono reel par tour
        if L == tour_arret: # On ajoute le temps d'un arret au stand 
            chrono_reel += temps_stands
        temps_total_course += chrono_reel
    return temps_total_course

# L'optimisation, on va tester tous les arrets possible entre le tours 10 et 40
resultats_strategies = {}
N_simulations = 1000 # Nb de simulations

for strategie in range(10, 40):
    temps_cumules = 0.0
    for i in range(N_simulations): # On simule 1000 fois cette strategie
        temps_cumules += simuler_course(strategie)
    temps_moyen_strategie = temps_cumules / N_simulations # On calcul la moyenne des 1000 courses
    resultats_strategies[strategie] = temps_moyen_strategie

# Analyse des resultats
meilleur_tour = min(resultats_strategies, key=resultats_strategies.get)
meilleur_temps = resultats_strategies[meilleur_tour]

print(f"fenetre d'arret optimale : {meilleur_tour}")
print(f"temps de course moyen : {meilleur_temps:.4f}s")




fenetre d'arret optimale : 28
temps de course moyen : 5498.9778s
